In [217]:
import pandas as pd
import numpy as np
import os 
import statsmodels.formula.api as smf
path = "D:\\other\\Dana\\traffic crashes\\chicago"
os.chdir(path)

In [218]:
df2 = pd.read_csv("chicago_crashes_cleaned_v2.csv", low_memory=False)


In [219]:
df2.columns

Index(['SEX', 'AGE', 'SAFETY_EQUIPMENT', 'INJURY_CLASSIFICATION',
       'AIRBAG_DEPLOYED', 'DRIVER_ACTION', 'DRIVER_VISION', 'BAC_RESULT VALUE',
       'CELL_PHONE_USE', 'MAKE', 'MODEL', 'VEHICLE_YEAR', 'VEHICLE_DEFECT',
       'MANEUVER', 'EXCEED_SPEED_LIMIT_I', 'FIRST_CONTACT_POINT'],
      dtype='object')

In [220]:
df2["SEX"].value_counts()

M    364586
F    259050
X       531
Name: SEX, dtype: int64

## Synthetic Dataset Generation

In [221]:
uv_IC = df2["INJURY_CLASSIFICATION"].unique()
uv_SB = df2["SAFETY_EQUIPMENT"].unique()
uv_DV = df2["DRIVER_VISION"].unique()
uv_G = df2["SEX"].unique()
uv_Ag = df2["AGE"].unique()
sample_size = 500000 #600000

In [222]:
c_IC = df2["INJURY_CLASSIFICATION"].value_counts()
d_IC = c_IC / len(df2)

c_SB = df2["SAFETY_EQUIPMENT"].value_counts()
d_SB = c_SB / len(df2)

c_DV = df2["DRIVER_VISION"].value_counts()
d_DV = c_DV / len(df2)

c_G = df2["SEX"].value_counts()
d_G = c_G / len(df2)

c_Ag = df2["AGE"].value_counts()
d_Ag = c_Ag / len(df2)


In [223]:
new_df = pd.DataFrame(columns=['IC', 'SB', 'DV', 'G', 'Ag', 'G_int'])
new_df['G'] = pd.Series(d_G.index).sample(n=sample_size, replace=True, weights=d_G.values).reset_index(drop=True)
new_df['G'].value_counts()

M    292263
F    207339
X       398
Name: G, dtype: int64

In [224]:
new_df['DV'] = pd.Series(d_DV.index).sample(n=sample_size, replace=True, weights=d_DV.values).reset_index(drop=True)
new_df['Ag'] = pd.Series(d_Ag.index).sample(n=sample_size, replace=True, weights=d_Ag.values).reset_index(drop=True)

In [225]:
mapping = {'M': 1, 'F': 2, 'X': 3}
new_df['G_int'] = new_df['G'].map(mapping)
new_df['G_int'].value_counts()

1    292263
2    207339
3       398
Name: G_int, dtype: int64

In [226]:
new_df["Ag"].max()

## Getting the effect

In [227]:
#import random


new_df['SB'] = np.random.binomial(n=1, p=((new_df['G_int']+new_df["Ag"])/(3+new_df["Ag"].max())), size=sample_size)
#new_df['SB'] = np.random.binomial(n=1, p=0.9, size=sample_size) # gives more accurate result

new_df['SB'] = (new_df['SB'] > 0).astype(np.int32)
new_df["IC"] = np.random.binomial(n=(5-(new_df['G_int']+new_df["Ag"])/(3+new_df["Ag"].max())-new_df['SB']), 
                                  p=0.4,  size=sample_size)



In [228]:
new_df['SB'].value_counts()

0    307734
1    192266
Name: SB, dtype: int64

In [229]:
new_df["IC"].value_counts()

1    189404
2    162060
0     81425
3     59234
4      7877
Name: IC, dtype: int64

## Applying DoWhy

In [230]:
import networkx as nx

G = nx.DiGraph()
nodes = ['IC', 'SB', 'DV', 'G_int','Ag']
G.add_nodes_from(nodes)
edges = [('G_int', 'SB'), ('Ag', 'SB'), ('DV', 'IC'),
         ('G_int', 'IC'), ('Ag', 'IC'), ('SB', 'IC'),]
G.add_edges_from(edges)
nx.write_gml(G, 'simple_graph.gml')


In [231]:
import logging
import dowhy
from dowhy import CausalModel
import dowhy.datasets
import econml
import warnings
warnings.filterwarnings('ignore')

In [232]:
graph_file = 'simple_graph.gml'

treatment_variable = 'SB'
outcome_variable = 'IC'

model = CausalModel(data=new_df,
                    treatment=treatment_variable, outcome=outcome_variable,
                    graph=graph_file)

In [233]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)
causal_estimate = model.estimate_effect(identified_estimand, method_name="backdoor.propensity_score_stratification")
print(causal_estimate)

Estimand type: nonparametric-ate

### Estimand : 1
Estimand name: backdoor
Estimand expression:
  d                 
─────(E[IC|Ag,Gᵢₙₜ])
d[SB]               
Estimand assumption 1, Unconfoundedness: If U→{SB} and U→IC then P(IC|SB,Ag,G_int,U) = P(IC|SB,Ag,G_int)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

propensity_score_stratification
*** Causal Estimate ***

## Identified estimand
Estimand type: nonparametric-ate

### Estimand : 1
Estimand name: backdoor
Estimand expression:
  d                 
─────(E[IC|Ag,Gᵢₙₜ])
d[SB]               
Estimand assumption 1, Unconfoundedness: If U→{SB} and U→IC then P(IC|SB,Ag,G_int,U) = P(IC|SB,Ag,G_int)

## Realized estimand
b: IC~SB+Ag+G_int
Target units: ate

## Estimate
Mean value: -0.4016776323709846

